In [ ]:
!pip install faiss-cpu sentence-transformers groq

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
from groq import Groq

In [ ]:
Groq.api_key = "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"

In [ ]:
docs =[
    "One of the 12 Jyotirlingas: The Kedarnath Temple is the highest of the 12 Jyotirlingas (supreme shrines dedicated to Lord Shiva) and is a primary destination in the sacred Chhota Char Dham Yatra.",
    "Mythological Origins: Legend dictates the temple was originally built by the Pandavas to seek Lord Shiva’s forgiveness for the sins committed during the Kurukshetra War. It was later restored by the sage Adi Shankaracharya in the 8th century.",
    "Epic Trek & Accessibility: Located 16-17 km from the nearest motorable point at Gaurikund, the shrine is accessible via a steep walking trek, ponies, or helicopter services.",
    "Survival of the 2013 Floods: The temple miraculously withstood the devastating 2013 flash floods in Uttarakhand. A massive boulder, known as the Bhim Shila, rolled down and wedged behind the temple, shielding it from destruction and debris.",
    "Seasonal Closure: Due to severe freezing conditions and heavy snowfall, the shrine is closed for six months every winter. During this time, the deity's idol is ceremoniously relocated and worshipped at the Omkareshwar Temple in Ukhimath."
]

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(docs).astype("float32")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
doc_embeddings.shape

(5, 384)

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [ ]:
query = "which is the highest of 12 jyotirlingas?"
query_embedding = model.encode([query]).astype("float32")
top_k = 2
distances,indices = index.search(query_embedding,top_k)
retrieved_chunks = [docs[i] for i in indices[0]]
print("Retrieved chunks:")
for chunk in retrieved_chunks:
  print("-",chunk)


Retrieved chunks:
- One of the 12 Jyotirlingas: The Kedarnath Temple is the highest of the 12 Jyotirlingas (supreme shrines dedicated to Lord Shiva) and is a primary destination in the sacred Chhota Char Dham Yatra.
- Seasonal Closure: Due to severe freezing conditions and heavy snowfall, the shrine is closed for six months every winter. During this time, the deity's idol is ceremoniously relocated and worshipped at the Omkareshwar Temple in Ukhimath.


In [ ]:
from groq import Groq
client = Groq(
    api_key="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"
)
content = "\n".join(retrieved_chunks)
prompt = f"""
Answer the question based only on the content below.
Context:
{content}
Question:
{query}
Answer:
"""
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2,
    max_tokens=200
)
print("Final Answer:\n")
print(response.choices[0].message.content)

Final Answer:

The Kedarnath Temple.
